In [2]:
!pip install openai pandas requests openpyxl

In [ ]:
OPENAI_API_KEY = your_key_here
SERPER_API_KEY=your_key_here

In [ ]:
# =========================================================
# C6 GROWTH SIGNAL PIPELINE
# Serper + OpenRouter(OpenAI SDK) + Python Scoring
# =========================================================

# INSTALL IF NEEDED:
# !pip install openai pandas requests openpyxl

import pandas as pd
import requests
import json
import time
from openai import OpenAI

# =========================================================
# CONFIG
# =========================================================

OPENROUTER_API_KEY=your_key_here
SERPER_API_KEY=your_key_here

MODEL_NAME = "openai/gpt-4o-mini"

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

# =========================================================
# LOAD DATA
# =========================================================

# companies.xlsx should contain:
# Company Name | Website

df = pd.read_excel("companies.xlsx")

# =========================================================
# SERPER SEARCH FUNCTION
# =========================================================

def serper_search(company_name, website=""):

    url = "https://google.serper.dev/search"

    query = f"""
    {company_name}
    specialty chemicals OR pharma intermediates OR manufacturing
    expansion certifications exports careers news GMP ISO
    """

    payload = {
        "q": query,
        "num": 8
    }

    headers = {
        "X-API-KEY": SERPER_API_KEY,
        "Content-Type": "application/json"
    }

    try:

        response = requests.post(
            url,
            headers=headers,
            json=payload,
            timeout=30
        )

        if response.status_code != 200:

            print(f"Serper Error {response.status_code} for {company_name}")

            return ""

        data = response.json()

        snippets = []

        for item in data.get("organic", [])[:8]:

            title = item.get("title", "")
            snippet = item.get("snippet", "")
            link = item.get("link", "")

            combined = f"""
TITLE: {title}
SNIPPET: {snippet}
LINK: {link}
"""

            snippets.append(combined)

        return "\n".join(snippets)

    except Exception as e:

        print(f"Serper Exception for {company_name}: {e}")

        return ""

# =========================================================
# OPENAI SIGNAL DETECTION
# =========================================================

def detect_growth_signals_ai(company_name, website, search_text):

    prompt = f"""
You are evaluating growth signals for a manufacturing company.

Company:
{company_name}

Website:
{website}

Search Evidence:
{search_text}

Evaluate whether these signals are:

- Strong
- Weak
- Absent

Signals:

1. Hiring
Examples:
careers page, job openings, recruitment, hiring

2. Facility Expansion
Examples:
new plant, expansion, upgraded manufacturing facility,
scale-up, capacity increase, new unit

3. Certifications
Examples:
WHO-GMP, USFDA, ISO, DSIR, approvals,
certified facility, halal certified

4. Active Website/News
Examples:
2024/2025 updates, news pages,
recent press releases, blogs, latest updates

5. Financial/Export Growth
Examples:
exports, export destinations, global markets,
international customers, revenue growth

IMPORTANT:
- Use reasonable business judgment
- Weak evidence is allowed
- Do NOT hallucinate
- Base answers ONLY on provided evidence
- Return ONLY valid JSON

Return STRICT JSON ONLY:

{{
  "Hiring": "Strong/Weak/Absent",
  "Facility Expansion": "Strong/Weak/Absent",
  "Certifications": "Strong/Weak/Absent",
  "Active Website": "Strong/Weak/Absent",
  "Financial/Export Growth": "Strong/Weak/Absent",
  "reason": ""
}}
"""

    try:

        response = client.chat.completions.create(

            model=MODEL_NAME,

            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a strict manufacturing research analyst. "
                        "Return only valid JSON."
                    )
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],

            temperature=0,
            max_tokens=400
        )

        text = response.choices[0].message.content.strip()

        # Remove markdown formatting
        text = text.replace("```json", "")
        text = text.replace("```", "")
        text = text.strip()

        # Robust JSON parsing
        try:

            result = json.loads(text)

        except:

            start = text.find("{")
            end = text.rfind("}") + 1

            cleaned = text[start:end]

            result = json.loads(cleaned)

        return result

    except Exception as e:

        print(f"OpenAI Error for {company_name}: {e}")

        return {
            "Hiring": "Absent",
            "Facility Expansion": "Absent",
            "Certifications": "Absent",
            "Active Website": "Absent",
            "Financial/Export Growth": "Absent",
            "reason": f"API Error: {str(e)}"
        }

# =========================================================
# SCORING FUNCTION
# =========================================================

def calculate_c6_score(result):

    signal_names = [
        "Hiring",
        "Facility Expansion",
        "Certifications",
        "Active Website",
        "Financial/Export Growth"
    ]

    detected_signals = []

    total_signal_strength = 0

    for signal in signal_names:

        value = result.get(signal, "Absent")

        if value == "Strong":

            total_signal_strength += 1

            detected_signals.append(
                f"{signal} (Strong)"
            )

        elif value == "Weak":

            total_signal_strength += 0.5

            detected_signals.append(
                f"{signal} (Weak)"
            )

    # FINAL C6 SCORE

    if total_signal_strength >= 2:
        c6_score = 20

    elif total_signal_strength >= 1:
        c6_score = 10

    else:
        c6_score = 0

    return (
        detected_signals,
        total_signal_strength,
        c6_score
    )

# =========================================================
# CONFIDENCE FUNCTION
# =========================================================

def get_confidence(total_signal_strength):

    if total_signal_strength >= 3:
        return "High"

    elif total_signal_strength >= 1:
        return "Medium"

    return "Low"

# =========================================================
# MAIN PIPELINE
# =========================================================

results = []

total_companies = len(df)

for idx, row in df.iterrows():

    company = str(row["Company Name"]).strip()

    website = ""

    if "Website" in df.columns:
        website = str(row["Website"]).strip()

    print(f"\n[{idx+1}/{total_companies}] Processing: {company}")

    # -----------------------------------------------------
    # STEP 1: SERPER SEARCH
    # -----------------------------------------------------

    search_text = serper_search(company, website)
    print(search_text[:500])

    # -----------------------------------------------------
    # HANDLE EMPTY SEARCH RESULTS
    # -----------------------------------------------------

    if len(search_text.strip()) < 50:

        results.append({
            "Company Name": company,
            "Website": website,
            "Growth Signals": "",
            "Signal Strength": 0,
            "C6 Score": 0,
            "Confidence": "Low",
            "Reason": "No search evidence retrieved",
            "Search Evidence": ""
        })

        continue

    # -----------------------------------------------------
    # STEP 2: AI SIGNAL DETECTION
    # -----------------------------------------------------

    ai_result = detect_growth_signals_ai(
        company,
        website,
        search_text[:1500]  # reduce token usage
    )

    # -----------------------------------------------------
    # STEP 3: PYTHON SCORING
    # -----------------------------------------------------

    (
        detected_signals,
        total_signal_strength,
        c6_score
    ) = calculate_c6_score(ai_result)

    confidence = get_confidence(
        total_signal_strength
    )

    # -----------------------------------------------------
    # STORE RESULTS
    # -----------------------------------------------------

    results.append({

        "Company Name": company,

        "Website": website,

        "Growth Signals":
            ", ".join(detected_signals),

        "Signal Strength":
            total_signal_strength,

        "C6 Score":
            c6_score,

        "Confidence":
            confidence,

        "Reason":
            ai_result.get("reason", ""),

        "Search Evidence":
            search_text[:2500]
    })

    # -----------------------------------------------------
    # RATE LIMIT PROTECTION
    # -----------------------------------------------------

    time.sleep(6)

# =========================================================
# SAVE OUTPUT
# =========================================================

output_df = pd.DataFrame(results)

output_df.to_excel(
    "c6_growth_scores.xlsx",
    index=False
)

print("\n===================================")
print("DONE")
print("Saved: c6_growth_scores.xlsx")
print("===================================")


[1/18] Processing: A R Life Sciences

TITLE: Pharmaceutical Intermediates in Hyderabad | A. R. Life Sciences
SNIPPET: Our units are ISO 9001:2015 and GMP certified, and one of our units is approved by the US Food and Drug Administration for intermediates to regulate compliance ...
LINK: https://www.arlifesciences.com/


TITLE: AR Lifesciences
SNIPPET: ISO 9001:2008 Certified. AR Lifesciences is an ISO certified Pharma company, committed to high quality products. We have also earned the right amount of ...
LINK: https://www.arlifesc

[2/18] Processing: lucent drugs

TITLE: Lucent Drugs: API & Intermediates Manufacturers
SNIPPET: Lucent Drugs is an integrated API manufacturing company located in Hyderabad, India. Our commitment to deliver the highest-quality API's and Intermediates is ...
LINK: https://www.lucentdrugs.com/


TITLE: About Us - Lucent Drugs
SNIPPET: Established in 2018, we are one of the top manufacturers of Tramadol and also enjoy a significant market share in the produc